In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import geopandas as gpd


In [ ]:

# file paths 
spm_novoa2017 = r"../data/Jul\S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_SPM_Novoa2017.tif"
spm_nechad2010 = r"../data/Jul\S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_SPM_Nechad2010_665.tif"
spm_nechad2016 = r"../data/Jul\S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_SPM_Nechad2016_665.tif"

hue_angle = r"../data/Jul\S2B_MSI_2025_07_30_05_03_55_T44QQE_BoB_L2W_hue_angle.tif"

In [ ]:

def read_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
        profile = src.profile
        nodata = src.nodata
    
    # mask nodata + huge fill values
    data[data == nodata] = np.nan
    data[data > 1e20] = np.nan   # ACOLITE fill value ~1e36
    data[data < 0] = np.nan
    
    return data, profile


spm10, prof = read_raster(spm_nechad2010)
spm16, _    = read_raster(spm_nechad2016)
spm17, _    = read_raster(spm_novoa2017)


In [ ]:
FILL = 9.96921e36  # ACOLITE fill value

# flatten and remove NaNs
mask = ~np.isnan(spm10) & ~np.isnan(spm16) & ~np.isnan(spm17)

x10 = spm10[mask]
x16 = spm16[mask]
x17 = spm17[mask]

#x10[x10 >= FILL/10] = np.nan
#x16[x16 >= FILL/10] = np.nan
#x17[x17 >= FILL/10] = np.nan

In [ ]:
def stats(x):
    return {
        "mean": np.nanmean(x),
        "median": np.nanmedian(x),
        "std": np.nanstd(x),
        "min": np.nanmin(x),
        "max": np.nanmax(x)
    }

print("Nechad 2010:", stats(x10))
print("Nechad 2016:", stats(x16))
print("Novoa 2017 :", stats(x17))

In [ ]:

plt.figure()
plt.scatter(x16, x10, s=1)
plt.xlabel("SPM Nechad 2016 (g m⁻3)")
plt.ylabel("SPM Nechad 2010 (g m⁻3)")
plt.title("Nechad 2016 vs Nechad 2010")
plt.ylim(0, 100)
plt.xlim(0, 100)
plt.show()


In [ ]:
diff_16_10 = spm16 - spm10
diff_17_16 = spm17 - spm16

flags_path = r"../data/Jul\S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_l2_flags.tif"   # or l2_flags.tif

with rasterio.open(flags_path) as src:
    flags = src.read(1).astype(np.int32)

# keep only water pixels
def has_flag(arr, bit):
    return (arr & (1 << bit)) != 0

mask_out = has_flag(flags, 4)
mask_cirrus     = has_flag(flags, 1)
mask_swir       = has_flag(flags, 0)
mask_negative   = has_flag(flags, 3)
mask_toa   = has_flag(flags, 2)

valid = ~(mask_out |  mask_toa  | mask_negative | mask_cirrus)

spm16 = np.where(valid, spm16, np.nan)
spm17 = np.where(valid, spm17, np.nan)


In [ ]:

masked_spm16_path =r"../data/Jul\S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_SPM_Nechad2016_665_masked.tif"

# Apply the mask
spm16_masked = np.where(valid, spm16, np.nan)

# Update the profile for the output GeoTIFF
output_profile = prof.copy()
output_profile.update(dtype=rasterio.float32, nodata=np.nan)

with rasterio.open(masked_spm16_path, 'w', **output_profile) as dst:
    dst.write(spm16_masked.astype(rasterio.float32), 1)

print(f"Masked SPM GeoTIFF saved to: {masked_spm16_path}")


In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(spm16_masked, vmax=40, cmap="turbo")
plt.colorbar(label="SPM (g m⁻3)")
plt.title("SPM Nechad 2016 (Masked)")
plt.show()


In [ ]:


masked_spm17_path =r"../data/Jul\S2B_MSI_2019_07_12_05_04_04_T44QQE_BoB_L2W_SPM_Novoa2017_masked.tif"

# Apply the mask
spm17_masked = np.where(valid, spm17, np.nan)

# Update the profile for the output GeoTIFF
output_profile = prof.copy()
output_profile.update(dtype=rasterio.float32, nodata=np.nan)

with rasterio.open(masked_spm17_path, 'w', **output_profile) as dst:
    dst.write(spm17_masked.astype(rasterio.float32), 1)

print(f"Masked SPM GeoTIFF saved to: {masked_spm17_path}")


In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(spm17_masked, vmax=40, cmap="turbo")
plt.colorbar(label="SPM (g m⁻3)")
plt.title("SPM Novoa 2017 (Masked)")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(spm, vmax=40, cmap="turbo")
plt.colorbar(label="SPM (g m⁻3)")
plt.title("Median SPM April")
plt.show()

In [ ]:
saturation_mask = (spm17 > 100) & (spm16 < 50)

plt.figure()
plt.imshow(saturation_mask)
plt.title("Likely Nechad Saturation Zones")
plt.show()


Histogram

In [ ]:
plt.figure()
#plt.hist(x10, bins=20, alpha=0.5, label="Nechad2010")
#plt.hist(x16, bins=20, alpha=0.5, label="Nechad2016")
plt.hist(x17, bins=20, alpha=0.5, label="Novoa2017")
plt.legend()
plt.xlabel("SPM (g m⁻3)")
plt.ylabel("Frequency")
plt.xlim(0,100)
plt.show()


In [ ]:
spm17_clipped = np.clip(spm17_masked, a_min=None, a_max=200)
    

In [ ]:
# 1. Gather all SPM files for the month
data_folder = r"../data/Jul"
spm_files = glob.glob(os.path.join(data_folder, "*SPM_Novoa2017*.tif"))

all_spm_data = []
for file in spm_files:
# Read, mask (using flags), and clip each file
        with rasterio.open(file) as src:
            data = src.read(1)
            profile = src.profile

        # Apply your masking logic here (matching flags to files)
        spm_masked = np.where(valid, data, np.nan)
        all_spm_data.append(spm_masked)

    # 2. Stack and calculate the median along the time axis
spm_stack = np.array(all_spm_data)
monthly_median_spm = np.nanmedian(spm_stack, axis=0)

# 3. Save the result
monthly_spm_path = os.path.join(data_folder, "monthly_median_spm_Novoa.tif")
output_profile = profile.copy()
output_profile.update(dtype=rasterio.float32, nodata=np.nan)

with rasterio.open(monthly_spm_path, 'w', **output_profile) as dst:
        dst.write(monthly_median_spm.astype(rasterio.float32), 1)

print(f"Monthly median SPM saved to: {monthly_spm_path}")